# Analyse des sentiments avec Word2Vec et Deep Learning

Dans le notebook 23, nous avons utilisé **TF-IDF** pour représenter les tweets, puis un Deep learning pour les classifier.

Dans ce notebook, nous explorons deux approches plus avancées :

| Approche | Représentation | Modèle | Avantages |
|---|---|---|---|
| TF-IDF + Deep learning (notebook 23) | Fréquence des mots (~14000 dims) | Réseau dense | Simple |
| **Word2Vec + Deep learning** | Moyenne des embeddings (300 dims) | Réseau dense | Sémantique des mots |
| **RNN + Word2Vec** | Séquence d'embeddings (300 dims) | RNN | Contexte et ordre |

**Pourquoi les embeddings sont-ils meilleurs que TF-IDF ?**
- "heureux" et "content" auront des vecteurs proches (sémantique)
- Dimensionnalité fixe et réduite : 300 vs ~14000 pour TF-IDF
- Le RNN capture en plus l'**ordre** et le **contexte** des mots

In [ ]:
!pip install gensim -q

# Importation des packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Importation des données

Ajoutez un raccourci de ces dossiers à votre Google Drive :

- Tweets nettoyés : https://drive.google.com/drive/folders/1uj-BnUzSHJOHuojQ9q8q53DN0EGiPpcl?usp=sharing
- Modèle Word2Vec : https://drive.google.com/drive/folders/1e6eRSPuZz3A3BXPVbggKHC_9XKU0MWAW?usp=sharing

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/RNN_sentiment_dataset/cleaned_tweets.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df = pd.concat([df.iloc[-50000:, :], df.iloc[:50000, :]])

In [ ]:
print(df['label'].value_counts())
print(df.shape)

# Chargement du modèle Word2Vec

Nous utilisons le modèle pré-entraîné de Google (vu dans le notebook 25) :
- Entraîné sur 100 milliards de mots (Google News)
- 3 millions de mots
- Vecteurs de 300 dimensions

In [ ]:
path_word2vec = '/content/drive/MyDrive/embeddings/GoogleNews-vectors-negative300.bin'
word2vec_model = KeyedVectors.load_word2vec_format(path_word2vec, binary=True)
print(f"Modèle chargé : {len(word2vec_model.key_to_index)} mots, dimension {word2vec_model.vector_size}")

---
# Partie 1 : Word2Vec + deep learning

## Représentation par moyenne des embeddings (Mean Pooling)

Pour chaque tweet, nous :
1. Divisons le texte en mots
2. Récupérons le vecteur Word2Vec de chaque mot connu
3. Calculons la **moyenne** de tous ces vecteurs → 1 vecteur de 300 dims

```
tweet : "great movie loved it"
         ↓        ↓      ↓    ↓
W2V : [v_great] [v_movie] [v_loved] [v_it]
                      ↓
              MOYENNE → vecteur 300 dims
```

Complétez la fonction `tweet_to_embedding` qui convertit un tweet en vecteur de 300 dimensions.

- Utilisez `model.key_to_index` pour vérifier si un mot est dans le vocabulaire
- Utilisez `model.get_vector(word)` pour récupérer le vecteur d'un mot
- Si aucun mot n'est connu, retournez un vecteur nul (`np.zeros`)
- Sinon, retournez la moyenne des vecteurs (`np.mean(..., axis=0)`)

In [ ]:
def tweet_to_embedding(tweet, model, embed_dim=300):
    """
    Convertit un tweet en vecteur d'embedding par moyenne des vecteurs de ses mots.

    Args:
        tweet (str): Le tweet nettoyé
        model: Le modèle Word2Vec (KeyedVectors)
        embed_dim (int): Dimension des embeddings

    Returns:
        np.ndarray: Vecteur de dimension embed_dim
    """
    words = tweet.split()
    vectors = []

    # TODO: Pour chaque mot, récupérez son vecteur s'il est dans le vocabulaire
    # Indice: utilisez model.key_to_index pour vérifier et model.get_vector() pour obtenir



    if len(vectors) == 0:
        return np.zeros(embed_dim)

    # TODO: Retournez la moyenne des vecteurs
    return None

## Application au jeu de données

Appliquez `tweet_to_embedding` à tous les tweets pour construire la matrice `X_embed` et le vecteur de labels `y`.

In [ ]:
%%time
# TODO: Construire X_embed (matrice de vecteurs) et y (labels)
# X_embed doit être un np.array de forme (n_samples, 300)

X_embed = None  # np.array([tweet_to_embedding(...) for ...])
y = None  # df['label'].values

In [ ]:
print(f"Forme de X_embed : {X_embed.shape}")
print(f"Forme de y : {y.shape}")
print(f"Distribution des classes : {dict(zip(*np.unique(y, return_counts=True)))}")

## Séparation entraînement / test

Divisez les données avec `train_test_split` (80% train, 20% test, `random_state=42`).

In [ ]:
# TODO: Séparer X_embed et y en jeux d'entraînement et de test
X_train, X_test, y_train, y_test = None

## Création des générateurs

Nous réutilisons la classe `CustomDataset` du notebook 23.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.input = x
        self.output = y

    def __len__(self):
        return len(self.output)

    def __getitem__(self, idx):
        return self.input[idx], self.output[idx]

In [ ]:
x_training = CustomDataset(
    torch.FloatTensor(np.float32(X_train)),
    torch.FloatTensor(np.expand_dims(np.float32(y_train), axis=-1))
)
x_testing = CustomDataset(
    torch.FloatTensor(np.float32(X_test)),
    torch.FloatTensor(np.expand_dims(np.float32(y_test), axis=-1))
)

Créez les `DataLoader` avec une taille de batch de **64**.

In [ ]:
# TODO: Créer le DataLoader d'entraînement (batch_size=64, shuffle=True)
dataloader_train = None

In [ ]:
for x, y_batch in dataloader_train:
    print(f"Forme du batch X : {x.shape}")  # (64, 300)
    print(f"Forme du batch y : {y_batch.shape}")  # (64, 1)
    break

In [ ]:
# TODO: Créer le DataLoader de test (batch_size=64, shuffle=False)
dataloader_test = None

## Fonctions d'entraînement

Nous réutilisons les fonctions `step` et `fit` du notebook 23.

In [ ]:
def number_of_good_prediction(prediction, target):
    one_hot_prediction = np.where(prediction > 0.5, 1, 0)
    return np.sum(one_hot_prediction == target)


def step(model, opt, criterion, x_train, y_train, metric_function):
    opt.zero_grad()
    prediction = model(x_train)
    loss = criterion(prediction, y_train)
    performance = metric_function(prediction.detach().numpy(), y_train.detach().numpy())
    loss.backward()
    opt.step()
    return model, loss, performance


def fit(model, optimizer, criterion, epoch, trainloader, testloader, metric_function):
    history_train_loss, history_test_loss = [], []
    history_train_metrics, history_test_metrics = [], []

    for e in range(epoch):
        train_loss_batch = test_loss_batch = 0
        train_metric_batch = test_metric_batch = 0

        model.train()
        for x_batch, y_batch in trainloader:
            model, train_loss, train_perf = step(model, optimizer, criterion, x_batch, y_batch, metric_function)
            train_loss_batch += train_loss.item()
            train_metric_batch += train_perf

        model.eval()
        with torch.no_grad():
            for x_batch, y_batch in testloader:
                prediction = model(x_batch)
                test_loss_batch += criterion(prediction, y_batch).item()
                test_metric_batch += metric_function(prediction.numpy(), y_batch.numpy())

        train_loss_batch /= len(trainloader.sampler)
        test_loss_batch /= len(testloader.sampler)
        train_metric_batch /= len(trainloader.sampler)
        test_metric_batch /= len(testloader.sampler)

        history_train_loss.append(train_loss_batch)
        history_test_loss.append(test_loss_batch)
        history_train_metrics.append(train_metric_batch)
        history_test_metrics.append(test_metric_batch)

        print(f'Epoch {e+1}/{epoch} | '
              f'loss: {train_loss_batch:.4f} | val_loss: {test_loss_batch:.4f} | '
              f'acc: {train_metric_batch:.4f} | val_acc: {test_metric_batch:.4f}')

    return model, history_train_loss, history_test_loss, history_train_metrics, history_test_metrics

## Modèle Deep Learning

Définissez un `nn.Sequential` avec l'architecture suivante :
- Couche linéaire : **300 → 128** neurones + ReLU
- Couche linéaire : **128 → 64** neurones + ReLU
- Couche linéaire : **64 → 1** neurone + Sigmoid

Comparez avec le notebook 23 : même architecture mais l'entrée passe de ~14000 à **300** grâce aux embeddings !

In [ ]:
# TODO: Définir le modèle Deep Learning avec nn.Sequential
model_mlp = None

In [ ]:
print(model_mlp)

## Optimiseur et fonction de coût

Utilisez `BCELoss` comme fonction de coût et `Adam` avec `lr=0.001` comme optimiseur.

In [ ]:
# TODO: Définir criterion et optimizer
criterion = None
optimizer = None

epoch = 10

## Entraînement

Utilisez la fonction `fit` pour entraîner le modèle. Stockez les résultats dans des variables avec le suffixe `_mlp`.

In [ ]:
# TODO: Entraîner model_mlp avec la fonction fit
model_mlp, hist_train_loss_mlp, hist_test_loss_mlp, hist_train_acc_mlp, hist_test_acc_mlp = None

## Visualisation des courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist_train_loss_mlp, label='Train')
axes[0].plot(hist_test_loss_mlp, label='Validation')
axes[0].set_title('Loss - W2V + MLP')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist_train_acc_mlp, label='Train')
axes[1].plot(hist_test_acc_mlp, label='Validation')
axes[1].set_title('Accuracy - W2V + MLP')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Évaluation

Calculez les prédictions sur le jeu de test et affichez :
- La matrice de confusion
- Le rapport de classification
- L'accuracy (stockez-la dans `mlp_accuracy`)

In [ ]:
# TODO: Calculer les prédictions avec model_mlp
# Attention: utilisez torch.no_grad() et convertissez X_test en FloatTensor
predictions_mlp = None
predictions_mlp_hot = np.where(predictions_mlp > 0.5, 1, 0)

print("Matrice de confusion :")
print(None)

print("\nRapport de classification :")
print(None)

mlp_accuracy = None
print(f"Accuracy W2V + MLP : {mlp_accuracy:.4f}")

---
# Partie 2 : RNN avec Embeddings

## Pourquoi le RNN ?

Avec le mean pooling, nous **perdons l'ordre** des mots :
- "Le film n'est **pas** bon" → même représentation que "Le film est **pas** non bon"

Le RNN traite la séquence de mots **dans l'ordre**, ce qui lui permet de capturer :
- La négation : "**pas** content"
- Le contexte : "c'est **vraiment** super"
- Les dépendances longue distance

```
tweet :  "great"  →  "movie"  →  "loved"  →  "it"
           ↓           ↓           ↓           ↓
RNN:    [h₁]  →   [h₂]   →   [h₃]   →   [h₄]  → classification
```

## Construction du vocabulaire

Pour le RNN, chaque tweet est représenté par une **séquence d'indices** (un entier par mot).

Nous construisons un vocabulaire à partir du corpus :
- Index 0 : token `<PAD>` (rembourrage pour égaliser la longueur des séquences)
- Index 1 : token `<UNK>` (mots inconnus)
- Index 2+ : mots du vocabulaire

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_SEQ_LEN = 50
EMBED_DIM = 300
HIDDEN_DIM = 128

# Compter tous les mots du corpus
all_words = []
for tweet in df['tweet']:
    all_words.extend(str(tweet).split())

counter = Counter(all_words)

# Construire le vocabulaire : <PAD>=0, <UNK>=1, puis les MAX_VOCAB_SIZE mots les plus fréquents
vocab = ['<PAD>', '<UNK>'] + [word for word, _ in counter.most_common(MAX_VOCAB_SIZE - 2)]
word2idx = {word: idx for idx, word in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

print(f"Taille du vocabulaire : {VOCAB_SIZE}")
print(f"Mots les plus fréquents : {counter.most_common(5)}")

## Matrice d'embedding initialisée avec Word2Vec

Pour chaque mot du vocabulaire, si ce mot existe dans Word2Vec, nous copions son vecteur dans la matrice d'embedding. Sinon, le vecteur reste nul (sera appris pendant l'entraînement).

In [ ]:
embed_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
n_found = 0

for word, idx in word2idx.items():
    if word in word2vec_model.key_to_index:
        embed_matrix[idx] = word2vec_model.get_vector(word)
        n_found += 1

print(f"Mots avec embedding pré-entraîné : {n_found}/{VOCAB_SIZE} ({100*n_found/VOCAB_SIZE:.1f}%)")

## Conversion des tweets en séquences d'indices

Chaque tweet est converti en une liste d'entiers :
- Tronqué à `MAX_SEQ_LEN` mots si trop long
- Complété par des 0 (`<PAD>`) si trop court

In [ ]:
def tweet_to_sequence(tweet, word2idx, max_len):
    words = str(tweet).split()[:max_len]
    seq = [word2idx.get(word, 1) for word in words]  # 1 = <UNK>
    seq = seq + [0] * (max_len - len(seq))  # 0 = <PAD>
    return seq

In [ ]:
%%time
X_seq = np.array([tweet_to_sequence(t, word2idx, MAX_SEQ_LEN) for t in df['tweet']])
print(f"Forme de X_seq : {X_seq.shape}")  # (n_samples, MAX_SEQ_LEN)

In [ ]:
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq, df['label'].values, test_size=0.2, random_state=42
)

## Dataset pour séquences

Créez la classe `SequenceDataset` qui hérite de `Dataset`.

**Différence avec `CustomDataset` :**
- Les séquences doivent être des `torch.LongTensor` (indices entiers, pas des flottants)
- Les labels sont des `torch.FloatTensor` avec une dimension supplémentaire (`unsqueeze(1)`)

In [ ]:
class SequenceDataset(Dataset):
    """
    Dataset pour les séquences de mots (indices entiers).
    """
    def __init__(self, sequences, labels):
        # TODO: Convertir sequences en torch.LongTensor
        # TODO: Convertir labels en torch.FloatTensor et ajouter une dimension avec unsqueeze(1)
        self.sequences = None
        self.labels = None

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # TODO: Retourner la séquence et le label à l'index idx
        return None, None

In [ ]:
ds_train = SequenceDataset(X_train_seq, y_train_seq)
ds_test = SequenceDataset(X_test_seq, y_test_seq)

In [ ]:
# TODO: Créer dl_train_rnn (batch_size=64, shuffle=True)
dl_train_rnn = None

In [ ]:
for x, y_batch in dl_train_rnn:
    print(f"Forme X : {x.shape}, dtype : {x.dtype}")  # (64, 50) torch.int64
    print(f"Forme y : {y_batch.shape}, dtype : {y_batch.dtype}")  # (64, 1) torch.float32
    break

In [ ]:
# TODO: Créer dl_test_rnn (batch_size=64, shuffle=False)
dl_test_rnn = None

## Modèle RNN

Créez la classe `SentimentRNN` avec l'architecture suivante :

```
Input (batch, 50)  ← séquences d'indices
    ↓
Embedding (batch, 50, 300)  ← nn.Embedding initialisée avec W2V
    ↓
RNN (batch, 50, 128)  ← hidden state de dimension 128
    ↓
Dernier hidden state (batch, 128)
    ↓
Linear (batch, 1)
    ↓
Sigmoid → probabilité entre 0 et 1
```

**Conseils :**
- `nn.Embedding(vocab_size, embed_dim, padding_idx=0)` : la couche d'embedding ignore les tokens `<PAD>`
- `self.embedding.weight.data.copy_(torch.FloatTensor(embed_matrix))` : initialise avec W2V
- `nn.RNN(embed_dim, hidden_dim, batch_first=True)` : `batch_first=True` pour avoir (batch, seq, features)
- Le RNN retourne `output, (hidden, cell)`. On veut le `hidden` final : forme `(1, batch, hidden_dim)`
- `hidden.squeeze(0)` retire la première dimension → `(batch, hidden_dim)`

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, embed_matrix=None):
        super().__init__()

        # Couche d'embedding (padding_idx=0 pour ignorer les tokens PAD)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Initialisation avec les vecteurs Word2Vec
        if embed_matrix is not None:
            self.embedding.weight.data.copy_(torch.FloatTensor(embed_matrix))

        # TODO: Définir self.rnn (nn.rnn avec batch_first=True)
        self.rnn = None

        # TODO: Définir self.fc (couche linéaire hidden_dim → 1)
        self.fc = None

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (batch_size, seq_len) - indices entiers

        # TODO: Passer x dans l'embedding
        # embedded: (batch_size, seq_len, embed_dim)
        embedded = None

        # TODO: Passer embedded dans le rnn
        # Récupérer seulement le hidden state final
        # hidden: (num_layers=1, batch_size, hidden_dim)

        # torch.max sur la dimension temporelle
        pooled, _ = None

        # TODO: Appliquer fc et sigmoid sur hidden.squeeze(0)

        # Retourner le résultat
        return None

In [ ]:
# TODO: Instancier le modèle avec VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM et embed_matrix
model_rnn = None

print(model_rnn)

In [ ]:
# Compter les paramètres
total_params = sum(p.numel() for p in model_rnn.parameters())
trainable_params = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)
print(f"\nTotal paramètres : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")

In [ ]:
model_rnn.embedding.weight.requires_grad = False

In [ ]:
# Compter les paramètres
total_params = sum(p.numel() for p in model_rnn.parameters())
trainable_params = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)
print(f"\nTotal paramètres : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")

## Entraînement du RNN

In [ ]:
# TODO: Définir criterion_rnn et optimizer_rnn (même paramètres que pour le MLP)
criterion_rnn = None
optimizer_rnn = None

In [ ]:
# TODO: Entraîner model_rnn avec la fonction fit
# Stockez les résultats dans des variables avec le suffixe _rnn
model_rnn, hist_train_loss_rnn, hist_test_loss_rnn, hist_train_acc_rnn, hist_test_acc_rnn = None

## Visualisation des courbes du RNN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist_train_loss_rnn, label='Train')
axes[0].plot(hist_test_loss_rnn, label='Validation')
axes[0].set_title('Loss - RNN + W2V')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(hist_train_acc_rnn, label='Train')
axes[1].plot(hist_test_acc_rnn, label='Validation')
axes[1].set_title('Accuracy - RNN + W2V')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Évaluation du RNN

In [ ]:
# TODO: Calculer les prédictions du RNN sur X_test_seq
# Attention: utilisez torch.no_grad() et convertissez X_test_seq en LongTensor
with torch.no_grad():
    preds_rnn = None

preds_rnn_hot = np.where(preds_rnn > 0.5, 1, 0)

print("Matrice de confusion :")
print(None)

print("\nRapport de classification :")
print(None)

rnn_accuracy = None
print(f"Accuracy RNN + W2V : {rnn_accuracy:.4f}")

---
# Partie 3 : Comparaison des approches

In [ ]:
# Comparaison visuelle des courbes d'apprentissage
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
axes[0].plot(hist_test_loss_mlp, label='W2V + Deep learning', marker='o')
axes[0].plot(hist_test_loss_rnn, label='RNN + W2V', marker='s')
axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(hist_test_acc_mlp, label='W2V + Deep learning', marker='o')
axes[1].plot(hist_test_acc_rnn, label='RNN + W2V', marker='s')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*65)
print("RÉSUMÉ DE LA COMPARAISON DES APPROCHES")
print("="*65)

results = pd.DataFrame({
    'Approche': ['TF-IDF + Deep learning  (notebook 23)', 'W2V Mean Pooling + Deep learning', 'RNN + W2V'],
    'Input dim': ['~14 000', '300', '50 (séquence)'],
    'Test Accuracy': ['~0.73 (référence)', f'{mlp_accuracy:.4f}', f'{rnn_accuracy:.4f}'],
    'Capture la sémantique': ['Non', 'Oui', 'Oui'],
    'Capture l\'ordre': ['Non', 'Non', 'Oui']
})

print(results.to_string(index=False))

## Discussion

**Observations attendues :**

1. **W2V + MLP vs TF-IDF + MLP** : L'embedding réduit drastiquement la dimensionnalité (300 vs ~14000) tout en capturant la sémantique. Le modèle généralise mieux car il a vu des relations entre mots pendant le pré-entraînement.

2. **RNN vs Deep learning** : Le RNN capture l'ordre des mots, ce qui est crucial pour comprendre la négation ("not good" vs "good not") et le contexte. Il devrait obtenir les meilleures performances.